# Baltimore Marriage Certificates — Structured Extraction (Step 2)

Turn OCR text (225 pages) into validated `MarriageRecord` JSON via an LLM, then
**compare two OCR engines** (Docling/RapidOCR vs TrOCR) feeding the same extractor.

**Pipeline:** OCR text → LLM with structured output → `artifacts/extracted/marriage_records_<source>.jsonl`

**Two OCR sources, one notebook:** set `OCR_SOURCE` to `"docling"` or `"trocr"` in
the config cell and run the batch. Each source writes its own checkpoint, so run
once per source, then use the final comparison cell to diff them.

**Resumable:** every extracted record is checkpointed to JSONL immediately; re-running the loop skips pages already done.

**Setup — API key (choose one provider):**
1. Create a `.env` file at the project root with your key:
   - OpenAI: `OPENAI_API_KEY=sk-...`
   - Gemini (free tier at https://aistudio.google.com/apikey): `GEMINI_API_KEY=...` (also run `uv add langchain-google-genai`)
2. Set `PROVIDER` in the config cell below.

In [26]:
# 1. Config & imports
import json
import sys
from pathlib import Path
import os

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))  # so `app.schema` imports work

# override=True so re-running this cell picks up a freshly-edited .env
load_dotenv(PROJECT_ROOT / ".env", override=True)
_key = os.getenv("OPENAI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if _key:
    print(f"API key loaded: {_key[:7]}...{_key[-4:]} (len {len(_key)})")
else:
    print("WARNING: no OPENAI_API_KEY / GOOGLE_API_KEY found in .env")

from app.schema import MarriageRecord  # noqa: E402

DOCLING_DIR = PROJECT_ROOT / "artifacts/docling"
TROCR_PATH = PROJECT_ROOT / "artifacts/TrOCR/trocr_output.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "artifacts/extracted"
# CHECKPOINT_PATH is derived from OCR_SOURCE in the config cell below.

API key loaded: sk-proj...HCsA (len 164)


In [27]:
# --- LLM config ---
PROVIDER = "openai"          # "openai" or "gemini"
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.0            # deterministic extraction


# --- OCR source switch ---
# Run the batch once with "docling", once with "trocr"; each writes its own
# checkpoint file. The final cell compares the two.

OCR_SOURCE = "trocr"       # "docling" or "trocr"

ENGINE_TAGS = {
    "docling": "docling-rapidocr",
    "trocr": "microsoft/trocr-base-printed",
}
OCR_ENGINE_TAG = ENGINE_TAGS[OCR_SOURCE]
CHECKPOINT_PATH = OUTPUT_DIR / f"marriage_records_{OCR_SOURCE}.jsonl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OCR source: {OCR_SOURCE}  (engine tag: {OCR_ENGINE_TAG})")
print(f"Output: {CHECKPOINT_PATH}")

OCR source: trocr  (engine tag: microsoft/trocr-base-printed)
Output: /Users/akhileshmadyastha/Desktop/projects/document-understanding/artifacts/extracted/marriage_records_trocr.jsonl


In [13]:
# 2. Load OCR text for the selected source: {(pdf_name, page_no): text}
from docling_core.types.doc import DoclingDocument


def load_pages(source: str) -> dict[tuple[str, int], str]:
    """Load per-page OCR text for the given source into {(pdf, page): text}."""
    pages: dict[tuple[str, int], str] = {}

    if source == "docling":
        for json_path in sorted(DOCLING_DIR.glob("*_content.json")):
            doc = DoclingDocument.load_from_json(json_path)
            pdf_name = json_path.stem + ".pdf"  # e.g. 1_content.pdf
            for page_no in sorted(doc.pages):
                pages[(pdf_name, page_no)] = doc.export_to_text(page_no=page_no)

    elif source == "trocr":
        # TrOCR JSONL records use the key "page" (not "page_no").
        with TROCR_PATH.open() as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                pages[(rec["source_pdf"], rec["page"])] = rec["text"]

    else:
        raise ValueError(f"Unknown OCR_SOURCE: {source!r}")

    return pages


pages = load_pages(OCR_SOURCE)
print(f"Loaded {len(pages)} pages from source '{OCR_SOURCE}'")
sample_key = sorted(pages)[0]
print(f"\nSample ({sample_key[0]} page {sample_key[1]}):\n")
print(pages[sample_key][:500])

Loaded 225 pages from 2 PDFs

Sample (1_content.pdf page 1):

GOOD ONLY IN-BALTIMORE CITY

Marriage License STATE OF MARYLAND

BALTIMORE CITY

No. 34178

APPLI  5221

IHereby Certify, That on this Jo day ot Aug

Nineteen hundred and thirty

in accordance with the License issued by the

Clerk of the Court of Common Pleas herein indicated.

JAMES S CRAWFORD

and ALLIME POYAS

(Name of troom)

(Name of Bride)

Residence of Groom BALTO MD

Age 29 Color COLNativity TENor country)

Occupation MUSICIAN

and who is.

(State here widower or divorced as the case may


In [28]:
# 3. LLM with structured output (provider-agnostic)
def build_llm():
    if PROVIDER == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError(
                "OPENAI_API_KEY not set. Add it to .env and RE-RUN cell 1 "
                "(load_dotenv) before this cell."
            )
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE, api_key=api_key)
    if PROVIDER == "gemini":
        api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError(
                "GOOGLE_API_KEY not set. Add it to .env and RE-RUN cell 1 "
                "(load_dotenv) before this cell."
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=GEMINI_MODEL, temperature=TEMPERATURE, google_api_key=api_key
        )
    raise ValueError(f"Unknown provider: {PROVIDER}")


llm = build_llm()
extractor = llm.with_structured_output(MarriageRecord)
print(f"LLM ready: {PROVIDER}")

LLM ready: openai


In [29]:
# 4. Extraction prompt (verbatim mode — no OCR correction)
SYSTEM_PROMPT = """\
You are a genealogical records analyst. You extract structured data from
OCR transcriptions of 1930s-1940s Baltimore City (Maryland) marriage licenses.

The OCR text is NOISY: characters are misread, field labels bleed into values,
and lines appear out of order. Use the form's known layout to decide WHICH field
each fragment belongs to — but copy the text exactly as it appears in the OCR.

Form layout (for field assignment only):
- Header: "GOOD ONLY IN BALTIMORE CITY", "STATE OF MARYLAND", "BALTIMORE CITY",
  license number ("No. ..."), application number ("APPLICATION # ..." / "APPLI ...").
- Certification: "I Hereby Certify, That on this <day> day of <month>,
  Nineteen hundred and <year words>", married at <location>.
- Groom block, then bride block. Each has: name, residence, age, color
  (WHT/COL), nativity (birth state/country), occupation, marital status
  (single/widower/widow/divorced).
- Consent lines if either party is a minor; relationship line
  (e.g. "NOT RELATED", "NONE"); officiant signature, official character, address.

Rules:
1. Extract values AS THEY APPEAR in the OCR text. Do NOT correct spelling,
   expand abbreviations, guess missing digits, or infer names/years/dates.
   Example: if OCR says "ALLIME POYAS", output "ALLIME POYAS" — not "ALLINE".
2. If OCR merges label and value (e.g. "COLNativity TEN", "OccupationONE"),
   split only when the boundary is unambiguous. Otherwise put the verbatim
   fragment in the best-matching field and leave related fields null.
3. Do NOT invent values. If a field is blank, illegible, or missing in the OCR,
   use null.
4. Preserve capitalization and punctuation exactly as in the OCR source.
5. Occupation "NONE" on the form is a real value — keep it as "NONE".
6. For dates: copy day/month/year fragments verbatim. Put the full year phrase
   in marriage_year_text. Set marriage_year only if all four digits appear
   explicitly in the OCR (do not infer "1939" from "Nineteen hundred and thirty").
"""

USER_PROMPT_TEMPLATE = """\
Extract the marriage record from this OCR transcription.
Copy values verbatim — do not correct OCR errors.

--- OCR TEXT START ---
{ocr_text}
--- OCR TEXT END ---
"""

print("Prompt ready (verbatim extraction — no OCR correction)")

Prompt ready (verbatim extraction — no OCR correction)


In [30]:
# 5. Smoke test: extract ONE page before running the full batch
test_key = sorted(pages)[0]
record = extractor.invoke([
    ("system", SYSTEM_PROMPT),
    ("user", USER_PROMPT_TEMPLATE.format(ocr_text=pages[test_key])),
])

record.source_pdf = test_key[0]
record.source_page = test_key[1]
record.ocr_engine = OCR_ENGINE_TAG

print(f"[{OCR_SOURCE}] {test_key}")
print(record.model_dump_json(indent=2))

[trocr] ('1_content.pdf', 1)
{
  "jurisdiction_note": "GOOD ONLY IN-BALTIMORE CITY",
  "state": "MARYLAND",
  "city": "BALTIMORE CITY",
  "license_number": "No. 34178",
  "application_number": "APPLI  5221",
  "marriage_day": null,
  "marriage_month": "Aug",
  "marriage_year": null,
  "marriage_year_text": "Nineteen hundred and thirty",
  "marriage_location": null,
  "groom": {
    "full_name": "JAMES S CRAWFORD",
    "residence": "BALTO MD",
    "age": 29,
    "color": "COL",
    "nativity": "TENor country",
    "occupation": "MUSICIAN",
    "marital_status": "widower"
  },
  "bride": {
    "full_name": "ALLIME POYAS",
    "residence": "DALTO D",
    "age": null,
    "color": null,
    "nativity": null,
    "occupation": "ONE",
    "marital_status": null
  },
  "groom_consent_by": null,
  "bride_consent_by": null,
  "parents_consent_filed": null,
  "relationship_between_parties": null,
  "officiant": {
    "signature": "J. M. Harris",
    "official_character": null,
    "address": nul

In [31]:
# 6. Full batch with resumable checkpoint.
# Each record is appended to the JSONL as soon as it's extracted; if the kernel
# dies or an API call fails, just re-run this cell - done pages are skipped.
import time


def load_checkpoint(path: Path) -> dict[tuple[str, int], dict]:
    """Rebuild {(pdf, page): record_dict} from an existing checkpoint."""
    done: dict[tuple[str, int], dict] = {}
    if not path.exists():
        return done
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            done[(rec["source_pdf"], rec["source_page"])] = rec
    return done


def extract_page(pdf_name: str, page_no: int, ocr_text: str) -> MarriageRecord:
    record = extractor.invoke([
        ("system", SYSTEM_PROMPT),
        ("user", USER_PROMPT_TEMPLATE.format(ocr_text=ocr_text)),
    ])
    record.source_pdf = pdf_name
    record.source_page = page_no
    record.ocr_engine = OCR_ENGINE_TAG
    return record


MAX_RETRIES = 3

records = load_checkpoint(CHECKPOINT_PATH)
if records:
    print(f"Resuming: {len(records)}/{len(pages)} pages already extracted\n")

with CHECKPOINT_PATH.open("a") as sink:
    for i, ((pdf_name, page_no), ocr_text) in enumerate(sorted(pages.items()), 1):
        if (pdf_name, page_no) in records:
            continue

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                record = extract_page(pdf_name, page_no, ocr_text)
                break
            except Exception as exc:
                if attempt == MAX_RETRIES:
                    print(f"  FAILED {pdf_name} p{page_no}: {exc}")
                    record = None
                else:
                    wait = 2**attempt  # 2s, 4s - handles rate limits
                    print(f"  retry {attempt} for {pdf_name} p{page_no} in {wait}s: {exc}")
                    time.sleep(wait)

        if record is None:
            continue

        rec_dict = record.model_dump()
        records[(pdf_name, page_no)] = rec_dict
        sink.write(json.dumps(rec_dict) + "\n")
        sink.flush()

        print(f"[{len(records)}/{len(pages)}] {pdf_name} p{page_no}: "
              f"{rec_dict['groom']['full_name']} & {rec_dict['bride']['full_name']}")

print(f"\nDone: {len(records)}/{len(pages)} records in {CHECKPOINT_PATH}")

[1/225] 1_content.pdf p1: JAMES S CRAWFORD & ALLIME POYAS
[2/225] 1_content.pdf p2: LEROY W. CUSTER & DCROTHY ISZKIEWICZ
[3/225] 1_content.pdf p3: WILLIAM A" CURREN & MARY HAJEK
[4/225] 1_content.pdf p4: CALVIN CURBEAM & MARY HOLTON
[5/225] 1_content.pdf p5: LALS E ERSON & None
[6/225] 1_content.pdf p6: PAUL ANDERSON & MARTHA LINOSAY
[7/225] 1_content.pdf p7: EVERETT VERNON ABBOTT & RUTH MARUE CAMPBELL
[8/225] 1_content.pdf p8: LESLIE ASHER & DORIS EPSTEIMde)
[9/225] 1_content.pdf p9: JOHN H. ADREON & HELEN MILLER
[10/225] 1_content.pdf p10: RAYMOND ANDERS & MARIE HINKLEMAN
[11/225] 1_content.pdf p11: JOSEPH C ACKMAN & ALBERTINA K TAYLOR
[12/225] 1_content.pdf p12: EDWRRD CHARLES APPEL & ANNETTE LOCILLE SCHAFFER
[13/225] 1_content.pdf p13: DAVID S ANDERSON & CLARA WATTS
[14/225] 1_content.pdf p14: VALENTIN ARRAS & FRANCES ANDREWS
[15/225] 1_content.pdf p15: MURRAY ABRAMSON & None
[16/225] 1_content.pdf p16: LEONARD U ALSRUHE & MARY doHANSON
[17/225] 1_content.pdf p17: EDGAR G. ABUGILL 

In [32]:
# 7. Quick quality check on the extracted batch
import pandas as pd

rows = list(records.values())
df = pd.json_normalize(rows)
print(f"Records: {len(df)}")

# Field fill-rates: which fields did the LLM find most/least often?
fill = df.notna().mean().sort_values(ascending=False)
print("\nField fill-rate (fraction non-null):")
print(fill.to_string(float_format=lambda x: f"{x:.2f}"))

# Spot-check a few extractions
df[["source_pdf", "source_page", "groom.full_name", "bride.full_name",
    "marriage_day", "marriage_month", "license_number"]].head(10)

Records: 225

Field fill-rate (fraction non-null):
jurisdiction_note              1.00
source_pdf                     1.00
state                          1.00
ocr_engine                     1.00
source_page                    1.00
groom.full_name                1.00
city                           1.00
groom.residence                0.99
bride.residence                0.98
bride.full_name                0.96
application_number             0.92
marriage_year_text             0.90
groom.marital_status           0.89
bride.marital_status           0.88
groom.age                      0.86
bride.occupation               0.85
groom.color                    0.84
license_number                 0.83
officiant.signature            0.81
groom.occupation               0.80
groom.nativity                 0.78
bride.color                    0.76
bride.age                      0.75
bride.nativity                 0.69
relationship_between_parties   0.68
marriage_location              0.42
marriage_mont

,source_pdf,source_page,groom.full_name,bride.full_name,marriage_day,marriage_month,license_number
0,1_content.pdf,1,JAMES S CRAWFORD,ALLIME POYAS,NaN,Aug,No. 34178
1,1_content.pdf,2,LEROY W. CUSTER,DCROTHY ISZKIEWICZ,5.0,ang,NaN
2,1_content.pdf,3,"WILLIAM A"" CURREN",MARY HAJEK,NaN,Aug,NaN
3,1_content.pdf,4,CALVIN CURBEAM,MARY HOLTON,NaN,NaN,No. 33871
4,1_content.pdf,5,LALS E ERSON,NaN,NaN,NaN,NaN
5,1_content.pdf,6,PAUL ANDERSON,MARTHA LINOSAY,NaN,Alugust,No. 34152
6,1_content.pdf,7,EVERETT VERNON ABBOTT,RUTH MARUE CAMPBELL,9.0,A,No. 34104
7,1_content.pdf,8,LESLIE ASHER,DORIS EPSTEIMde),3.0,NaN,No. 33779
8,1_content.pdf,9,JOHN H. ADREON,HELEN MILLER,7.0,aug,No.
9,1_content.pdf,10,RAYMOND ANDERS,MARIE HINKLEMAN,NaN,aug.,No. 33892


In [ ]:
# 8. (Optional) Reset checkpoint - uncomment to re-extract from scratch,
# e.g. after changing the model, prompt, or schema.

# CHECKPOINT_PATH.unlink(missing_ok=True)
# print("Checkpoint cleared")

In [33]:
# 9. Compare extractions across OCR sources.
# Run the batch once per OCR_SOURCE ("docling" then "trocr"), then run this cell.
import pandas as pd


def load_records(source: str) -> dict[tuple[str, int], dict]:
    path = OUTPUT_DIR / f"marriage_records_{source}.jsonl"
    out: dict[tuple[str, int], dict] = {}
    if not path.exists():
        return out
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            out[(r["source_pdf"], r["source_page"])] = r
    return out


docling_recs = load_records("docling")
trocr_recs = load_records("trocr")
print(f"docling: {len(docling_recs)} records | trocr: {len(trocr_recs)} records")

# Key fields to compare head-to-head (dotted path -> getter).
COMPARE_FIELDS = {
    "groom.full_name": lambda r: r["groom"]["full_name"],
    "bride.full_name": lambda r: r["bride"]["full_name"],
    "license_number": lambda r: r["license_number"],
    "marriage_month": lambda r: r["marriage_month"],
    "groom.age": lambda r: r["groom"]["age"],
    "bride.age": lambda r: r["bride"]["age"],
}

shared = sorted(set(docling_recs) & set(trocr_recs))
rows = []
for key in shared:
    d, t = docling_recs[key], trocr_recs[key]
    for field, get in COMPARE_FIELDS.items():
        dv, tv = get(d), get(t)
        rows.append({
            "pdf": key[0], "page": key[1], "field": field,
            "docling": dv, "trocr": tv, "agree": dv == tv,
        })

cmp = pd.DataFrame(rows)
if cmp.empty:
    print("\nNeed BOTH sources extracted. Run the batch once per OCR_SOURCE.")
else:
    print(f"\nComparing {len(shared)} pages present in both sources.")
    print("\nAgreement rate by field (docling == trocr):")
    print(cmp.groupby("field")["agree"].mean().to_string(float_format=lambda x: f"{x:.2f}"))
    print("\nSample disagreements:")
    display(cmp[~cmp["agree"]].head(15))

docling: 225 records | trocr: 225 records

Comparing 225 pages present in both sources.

Agreement rate by field (docling == trocr):
field
bride.age         0.98
bride.full_name   0.99
groom.age         0.98
groom.full_name   1.00
license_number    0.88
marriage_month    0.90

Sample disagreements:


,pdf,page,field,docling,trocr,agree
2,1_content.pdf,1,license_number,34178,No. 34178,False
7,1_content.pdf,2,bride.full_name,DOROTHY ISZKIEWICZ,DCROTHY ISZKIEWICZ,False
9,1_content.pdf,2,marriage_month,Aug,ang,False
20,1_content.pdf,4,license_number,33871,No. 33871,False
22,1_content.pdf,4,groom.age,20,None,False
23,1_content.pdf,4,bride.age,None,20,False
32,1_content.pdf,6,license_number,34152,No. 34152,False
33,1_content.pdf,6,marriage_month,August,Alugust,False
34,1_content.pdf,6,groom.age,22,21,False
35,1_content.pdf,6,bride.age,21,None,False
